# Visualize and test EM synaptome

In this notebook we select and download an `EM synaptome`. That is:
  - a skeleton representation of a single neuron from an electron microscopic dense tissue reconstruction
  - plus the spines on its surface, individually extracted
  - plus all afferent synapses onto the neuron, mapped to individual spines, shafts or the soma

This notebook serves as a starting point to a new user. It shows ways to access synapse and spine information. It provides a visualization of cell surface meshes, spines and afferent synapses together.


## Platform authentication
We begin by authenticating with the OBI platform to be able to access the synaptomes.

Please follow the instruction below to authenticate, then select the project to work with. 
The synaptome you want to visualize must be either public, or generated within that project.


In [ ]:
import os
import obi_auth
import numpy
import h5py
import tqdm
import bluepysnap as snap
from obi_notebook.get_projects import get_projects
from obi_notebook.get_entities import get_entities
from obi_notebook import get_environment
from entitysdk import Client
from entitysdk.models import CellMorphology, Subject, EMCellMesh, Circuit, TaskActivity, TaskConfig
from entitysdk.staging import stage_circuit
from morph_spines import load_morphology_with_spines
from ipywidgets import widgets

from pathlib import Path

environment = get_environment.get_environment()

token = obi_auth.get_token(environment=environment, auth_mode="daf")
project_context = get_projects(token, env=environment)

## Selecting a synaptome

**IMPORTANT. Read this carefully** 

`Synaptomes` is what we call simulatable models of a neuron and its afferents. There are many on the OBI platform, but not all of them have been generated from electron microscopy. Others have been built as statistical models by stochastic algorithms.

For this notebook, we require a `Synaptome` from electron microscopy. Below, you will be provided with a table to select a `Synaptome` from. Please make sure you select one where the value in the last column is `em_reconstruction`. Otherwise, in the next cell an error will be raised.


In [ ]:
client = Client(project_context=project_context, token_manager=token, environment=environment)

circs = client.search_entity(entity_type=Circuit, query={
    "scale": "single",
    "build_category": "em_reconstruction"
}).all()

select_circuit = widgets.Dropdown(options={
    circ.name: circ for circ in circs
})
display(select_circuit)

In [ ]:
select_circuit.value.description

## Download the `Synaptome`. Load neuron morphology and synapses


In [ ]:
circ_entity = client.get_entity(entity_id=select_circuit.value.id, entity_type=Circuit)
if circ_entity.build_category != "em_reconstruction":
    raise ValueError("The selected synaptome is not derived from an electron microscopic reconstruction!")

# This downloads the selected Synaptome to the local system.
circ_path = stage_circuit(client=client, model=circ_entity, output_dir=Path("downloaded_synaptome"))

# This loads the Synaptome as a `Circuit` object
circ = snap.Circuit(circ_path)
# Technically, the single neuron is represented as a `node population`. We access it.
nodes = [circ.nodes[node] for node in list(circ.nodes) if circ.nodes[node].type == "biophysical"]
virtuals = [circ.nodes[node] for node in list(circ.nodes) if circ.nodes[node].type == "virtual"]
if len(nodes) < 1:
    raise ValueError("No biophysical node population found!")
if len(virtuals) < 1:
    raise ValueError("No virtual afferent population found!")
node = nodes[0]
virtual = virtuals[0]
# All the presynaptic neurons innervating it are a second `node population`. We get their types.
afferents = virtual.get(properties="synapse_class")

# This loads the neuron along with the detected spines. Later we will see how to access spines.
m = load_morphology_with_spines(node.config["alternate_morphologies"]["h5v1"], load_meshes=True)

# Technically, the synapses are represented as an `edge population`. We access it here.
edge = circ.edges[list(circ.edges)[0]]
# We load the synapses of the synaptome neuron. First argument is an identifier of the neuron.
# Since a Synaptome contains only a single biophysical neuron, we just fill in 0.
syns = edge.afferent_edges(0, properties=edge.property_names)
syns["presyn_type"] = afferents[syns["@source_node"]].to_numpy()


## Print neuron info

Here, we simply print some basic information about the neuron we will visualize

In [ ]:
display(node.get())

# Generate spine and synapse colors

We generate colors for all spines and synapses.

For spines, we color them based on the number of synapses placed on them. Ranging from "light blue - no synapses" to "pink - many synapses".

For synapses it is red for spine synapes and white for shaft synapses.

In [ ]:
from matplotlib import cm

def pack_rgb(rgb_u8):
    """(N,3) uint8 → (N,) uint32  packed 0x00RRGGBB"""
    r = rgb_u8[:, 0].astype(numpy.uint32)
    g = rgb_u8[:, 1].astype(numpy.uint32)
    b = rgb_u8[:, 2].astype(numpy.uint32)
    return (r << 16) | (g << 8) | b
white = pack_rgb(numpy.array([[255, 255, 255]]).astype(numpy.uint8))[0]
red = pack_rgb(numpy.array([[255, 0, 0]]).astype(numpy.uint8))[0]

syns_per_spine = syns["spine_id"].value_counts().drop(-1).reindex(range(m.spines.spine_count)).fillna(0)
syns_per_spine_ = syns_per_spine / syns_per_spine.max()
spine_cols = pack_rgb((255 * cm.cool(syns_per_spine_)).astype(numpy.uint8))
syn_cols = numpy.array([red if spine_id > -1 else white for spine_id in syns["spine_id"]])

syns = syns.reset_index(drop=False).set_index("afferent_section_id")

# Visualize

Optionally, you can add a ghostly outline of the original input mesh to the visualization.

However, that will make the visualization slower and requires download of the mesh, which ma take a while.

If you prefer that option, set the following parameter to True.

If you do do, you can **optionally** provide the `entity id` of the mesh. Otherwise, the mesh will be looked up from the provenance of the Synaptome.

In [ ]:
SHOW_NEURON_MESH = True

mesh_entity_id = None

In [ ]:
vtx_surf = None

if SHOW_NEURON_MESH:
    import pylmesh
    from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID
    mesh_path = "downloaded_synaptome/source_mesh.glb"
    mesh_entity = None

    if mesh_entity_id is None:
        ta = client.search_entity(entity_type=TaskActivity, query={
            "generated__id": select_circuit.value.id
        }).one_or_none()
        if (ta is None) or (ta.used is None):
            pass
        else:
            cfg = client.get_entity(entity_id=ta.used[0].id, entity_type=TaskConfig)
            src_morph_entity = CellMorphologyFromID(id_str=str(cfg.inputs[0].id))
            mesh_entity = src_morph_entity.source_mesh_entity(db_client=client)
    else:
        mesh_entity = client.get_entity(entity_id=mesh_entity_id, entity_type=EMCellMesh)

    if mesh_entity is None:
        print("Source mesh cannot be resolved. Please continue without mesh viz.")
        SHOW_NEURON_MESH = False
    else:
        client.download_file(entity_id=mesh_entity.id, entity_type=EMCellMesh, asset_id=mesh_entity.assets[0].id, output_path=mesh_path)
        mesh_obj = pylmesh.load_mesh(str(mesh_path))
        vtx_surf = mesh_obj.vertices[::10]
        mesh_obj = None
        vtx_surf = (numpy.array([[v.x, v.y, v.z] for v in vtx_surf]) * 1E-3).astype(numpy.float32)


In [ ]:
import k3d 

plot_face = k3d.plot(background_color=0x000000, grid_visible=False, camera_auto_fit=False)

vtx = m.soma.soma_mesh_points.astype(numpy.float32)
fac = m.soma.soma_mesh_triangles.astype(numpy.uint32)
plot_face += k3d.mesh(
            vtx, fac,
            color=0x00FF00
            )
sec_line_dict = {}
added_spine_lst = []
added_syn_lst = []
for sec in m.morphology.sections:
    if SHOW_NEURON_MESH:
        w = 0.1
        col = 0xDDDD00
    else:
        w = float(sec.points[:, -1].mean())
        col = 0x00FF00
    sec_line_dict[sec.id + 1] = k3d.line(
        vertices=sec.points[:, :3],
        color=col,
        width=w,
        opacity=0.75,
        shader="mesh"
    )
    plot_face += sec_line_dict[sec.id + 1]

if vtx_surf is not None:
    plot_face += k3d.points(
        vtx_surf,
        point_size=0.15,
        opacity=0.5,
        color=0x00DD00,
        shader="points"
    )

def add_section_info(tgt_section):
    global plot_face
    tgt_sec = m.morphology.sections[tgt_section - 1]
    cam_tgt = tgt_sec.points[:, :3].mean(axis=0)
    cam_up = numpy.array([0, 1, 0])
    cam_cross = numpy.cross(cam_up, tgt_sec.points[-1, :3] - tgt_sec.points[0, :3])
    cam_cross = 25.0 * cam_cross / numpy.linalg.norm(cam_cross)
    cam_pos = cam_tgt + cam_cross
    cam_params = numpy.hstack([cam_pos, cam_tgt, cam_up])  

    while len(added_spine_lst) > 0:
        plot_face -= added_spine_lst.pop()
    while len(added_syn_lst) > 0:
        plot_face -= added_syn_lst.pop()

    plot_face.camera = cam_params
    for spine_id in m.spines.spine_table.set_index("afferent_section_id").loc[[tgt_section], "spine_id"]:
        vtx = m.spines.spine_mesh_points(spine_id).astype(numpy.float32)
        fac = m.spines.spine_mesh_triangles(spine_id).astype(numpy.uint32)
        if len(fac) > 0:
            spine = k3d.mesh(
            vtx, fac,
            color=int(spine_cols[spine_id]),
            opacity=0.5
            )
            plot_face += spine
            added_spine_lst.append(spine)

    if tgt_section in syns.index:
        sec_syns = syns.loc[[tgt_section]]
        for _, syn_row in sec_syns.iterrows():
            syn = k3d.points(
                syn_row[["afferent_synapse_x", "afferent_synapse_y", "afferent_synapse_z"]].to_numpy().astype(numpy.float32),
                colors=syn_cols[syn_row["index"]],
                point_size=0.5
            )
            plot_face += syn
            added_syn_lst.append(syn)

sel_sec = widgets.Dropdown(options={
    f"Section {sec}: {val} spines": sec for sec, val in m.spines.spine_table["afferent_section_id"].value_counts().items()
    })
plot_face.display()
display(widgets.interactive(add_section_info, tgt_section=sel_sec))